In [2]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

ModuleNotFoundError: No module named 'pandas'

Connexion PostgreSQL (rôle DS = lecture Silver + Gold)

In [ ]:
PG_CONN = (
    f"postgresql://ds_user1:{os.environ['DS_USER1_PASSWORD']}"
    f"@postgres:5432/esg_territorial"
)
engine = create_engine(PG_CONN)

Chargement de la table Gold finale

In [ ]:
print("Chargement gold.gold_esg_iris_final...")
df = pd.read_sql("SELECT * FROM gold.gold_esg_iris_final", engine)
print(f"  → {len(df):,} IRIS chargés")
print(f"  → Complétude moyenne: {df['pct_completude'].mean():.1f}%")
print(f"  → Score ESG moyen: {df['score_esg_0_100'].mean():.1f}/100")

Distribution des scores ESG

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes,
    ['score_env_0_100', 'score_social_0_100', 'score_gouvernance_0_100'],
    ['Score E', 'Score S', 'Score G']):
    df[col].dropna().hist(bins=30, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Score 0-100')
    ax.axvline(df[col].median(), color='red', linestyle='--', label=f'Médiane: {df[col].median():.0f}')
    ax.legend()
plt.suptitle("Distribution des scores ESG par IRIS", fontsize=14)
plt.tight_layout()
plt.savefig('/home/jovyan/work/outputs/distribution_scores_esg.png', dpi=150)
plt.show()

Corrélation pauvreté × chômage

In [ ]:
df_clean = df[['taux_pauvrete_60', 'taux_chomage_pct', 'bv_label']].dropna()
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_clean['taux_chomage_pct'], df_clean['taux_pauvrete_60'],
           alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('Taux de chômage (%)')
ax.set_ylabel('Taux de pauvreté seuil 60% (%)')
ax.set_title('Corrélation chômage × pauvreté par IRIS')
corr = df_clean['taux_pauvrete_60'].corr(df_clean['taux_chomage_pct'])
ax.text(0.05, 0.95, f'r = {corr:.2f}', transform=ax.transAxes, fontsize=12)
plt.tight_layout()
plt.savefig('/home/jovyan/work/outputs/scatter_chomage_pauvrete.png', dpi=150)
plt.show()

Top 10 IRIS les plus fragiles (score ESG le plus bas)

In [ ]:
fragiles = df.nsmallest(10, 'score_esg_0_100')[
    ['iris_code', 'iris_label', 'bv_label', 'score_esg_0_100',
     'taux_pauvrete_60', 'taux_chomage_pct', 'revenu_median_uc']
]
print("\nTop 10 IRIS les plus fragiles (score ESG le plus bas):")
print(fragiles.to_string(index=False))


Analyse Silver (feature pour ML)

In [ ]:
df_silver = pd.read_sql("""
    SELECT
        f.iris_code,
        f.revenu_median_uc,
        f.taux_pauvrete_60,
        f.ratio_interdecile_d9d1,
        a.taux_chomage_pct,
        a.taux_activite_pct,
        p.population_totale,
        p.part_moins_25_ans_pct,
        p.part_cadres_pct,
        l.part_logements_avant_1971_pct,
        l.part_hlm_pct,
        b.nb_equipements_total,
        b.nb_equip_sante
    FROM silver.silver_filosofi f
    LEFT JOIN silver.silver_rp_activite a USING (iris_code)
    LEFT JOIN silver.silver_rp_population p USING (iris_code)
    LEFT JOIN silver.silver_rp_logement l USING (iris_code)
    LEFT JOIN silver.silver_bpe b USING (iris_code)
""", engine)
print(f"\nDonnées Silver: {len(df_silver):,} IRIS, {len(df_silver.columns)} features")
print(df_silver.describe().T[['count','mean','std','min','max']])

engine.dispose()
print("\nAnalyse terminée. Outputs sauvegardés dans /home/jovyan/work/outputs/")